# 12 — Paper 2: Evaluasi Adversarial (clean / evasion / adaptive)

**Dijalankan di SageMaker, SETELAH notebook 11.** Mengevaluasi keempat varian model
(`baseline`, `fewshot`, `adv`, `fewshot_adv`) pada dua dimensi sekaligus:

1. **Generalisasi lintas-jaringan** (clean): MCC in-domain (source-test) dan cross (target-test).
2. **Ketahanan evasion**: MCC di bawah
   - `unconstrained` FGSM (batas atas daya serang, tak realistis),
   - `functional-preserving` FGSM (flow valid protokol — realistis, sesuai notebook 07),
   - `adaptive white-box` (saliency dari model target sendiri — skenario terburuk, notebook 08).

**Klaim yang diuji (Paper 2):** apakah `fewshot_adv` mempertahankan generalisasi lintas-jaringan
(seperti `fewshot`) SEKALIGUS lebih tahan evasion (seperti `adv`), atau ada trade-off.

Model dimuat dari `paper2_models/` (dibuat nb11) atau diunduh dari S3 `unsw-far/paper2/`.
Hasil + heatmap diupload ke S3 `unsw-far/paper2_eval/`.

In [ ]:
import importlib, subprocess, sys
for pkg,imp in [('pandas','pandas'),('numpy','numpy'),('scikit-learn','sklearn'),('xgboost','xgboost'),('matplotlib','matplotlib'),('boto3','boto3')]:
    try: importlib.import_module(imp)
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])
import pickle, os, json, glob
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import matthews_corrcoef, f1_score, accuracy_score
from xgboost import XGBClassifier
plt.rcParams.update({'figure.dpi':120,'font.size':9})
CIC_PKL='../../CICDDoS2018/data/cleaned_100.pkl'
UNSW_TRAIN='../data/UNSW_NB15_testing-set.csv'; UNSW_TEST='../data/UNSW_NB15_training-set.csv'
MODELDIR='paper2_models'; OUTDIR='paper2_eval_out'; os.makedirs(OUTDIR,exist_ok=True)
S3_BUCKET=os.environ.get('S3_BUCKET','ssh-detection-features-232032302717')
S3_PREFIX='unsw-far'; REGION=os.environ.get('AWS_REGION','ap-southeast-1')
SEED=42; H=0.01; EPS_EVAL=[0.05,0.1,0.2]; MAXN=40000
VARIANTS=['baseline','fewshot','adv','fewshot_adv']
print('=== SEL 0 (setup) SELESAI ===')

In [ ]:
# Unduh model dari S3 bila folder lokal belum ada.
if not os.path.isdir(MODELDIR) or not glob.glob(os.path.join(MODELDIR,'*','*.json')):
    try:
        import boto3; s3=boto3.client('s3',region_name=REGION)
        pfx=f'{S3_PREFIX}/paper2/'; os.makedirs(MODELDIR,exist_ok=True)
        pag=s3.get_paginator('list_objects_v2')
        for page in pag.paginate(Bucket=S3_BUCKET,Prefix=pfx):
            for o in page.get('Contents',[]):
                rel=o['Key'][len(pfx):]
                if not rel: continue
                lp=os.path.join(MODELDIR,rel); os.makedirs(os.path.dirname(lp),exist_ok=True)
                s3.download_file(S3_BUCKET,o['Key'],lp)
        print('model diunduh dari S3.')
    except Exception as e: print('unduh S3 gagal:',e)
print('dirs:', [d for d in glob.glob(os.path.join(MODELDIR,'*')) if os.path.isdir(d)])
print('=== SEL 1 (ambil model) SELESAI ===')

In [ ]:
# --- SFM + util + serangan (konsisten 07/08) ---
MAP_A={'duration':('Flow Duration','dur'),'fwd_pkts':('Tot Fwd Pkts','spkts'),
       'bwd_pkts':('Tot Bwd Pkts','dpkts'),'fwd_bytes':('TotLen Fwd Pkts','sbytes'),
       'bwd_bytes':('TotLen Bwd Pkts','dbytes'),'fwd_mean':('Fwd Pkt Len Mean','smean'),
       'bwd_mean':('Bwd Pkt Len Mean','dmean'),'src_load':('Flow Byts/s','sload'),
       'dst_load':('Bwd Pkts/s','dload')}
CANON=list(MAP_A.keys()); IX={c:i for i,c in enumerate(CANON)}
def build_matrix(df,side):
    idx=0 if side=='cic' else 1; cols=[MAP_A[c][idx] for c in CANON]
    out=df[cols].copy(); out.columns=CANON; out=out.replace([np.inf,-np.inf],np.nan)
    return out.fillna(out.median(numeric_only=True)).fillna(0.0).astype(float).values
def ev(yt,yp):
    return dict(mcc=float(matthews_corrcoef(yt,yp)),f1=float(f1_score(yt,yp,zero_division=0)),acc=float(accuracy_score(yt,yp)))
def loss_bin(model,X,y):
    p=np.clip(model.predict_proba(X)[:,1],1e-15,1-1e-15); y=y.astype(float)
    return -(y*np.log(p)+(1-y)*np.log(1-p))
def saliency(model,X,y,h=H):
    n,m=X.shape; S=np.zeros((n,m))
    for i in range(m):
        Xp=X.copy(); Xp[:,i]+=h; Xm=X.copy(); Xm[:,i]-=h
        S[:,i]=(loss_bin(model,Xp,y)-loss_bin(model,Xm,y))/(2*h)
    return S
def project_functional(Xadv_scaled, mean, scale):
    Xo=Xadv_scaled*scale+mean; Xo=np.clip(Xo,0.0,None)
    Xo[:,IX['fwd_pkts']]=np.round(Xo[:,IX['fwd_pkts']]); Xo[:,IX['bwd_pkts']]=np.round(Xo[:,IX['bwd_pkts']])
    Xo[:,IX['fwd_bytes']]=np.maximum(Xo[:,IX['fwd_bytes']],Xo[:,IX['fwd_pkts']])
    Xo[:,IX['bwd_bytes']]=np.maximum(Xo[:,IX['bwd_bytes']],Xo[:,IX['bwd_pkts']])
    with np.errstate(divide='ignore',invalid='ignore'):
        fm=np.where(Xo[:,IX['fwd_pkts']]>0,Xo[:,IX['fwd_bytes']]/Xo[:,IX['fwd_pkts']],0.0)
        bm=np.where(Xo[:,IX['bwd_pkts']]>0,Xo[:,IX['bwd_bytes']]/Xo[:,IX['bwd_pkts']],0.0)
    Xo[:,IX['fwd_mean']]=fm; Xo[:,IX['bwd_mean']]=bm
    return (Xo-mean)/scale
def fgsm_unconstrained(X,S,eps): return X+eps*np.sign(S)
def fgsm_functional(X,S,eps,mean,scale):
    Xadv=X+eps*np.sign(S); Xo=Xadv*scale+mean; Xr=X*scale+mean
    for j in [IX[c] for c in ['fwd_pkts','bwd_pkts','fwd_bytes','bwd_bytes','duration']]:
        Xo[:,j]=np.maximum(Xo[:,j],Xr[:,j])  # monotonic add-only
    return project_functional((Xo-mean)/scale, mean, scale)
print('=== SEL 2 (util + serangan) SELESAI ===')

In [ ]:
# --- Muat data uji (test set) + scaler per dataset ---
with open(CIC_PKL,'rb') as f: d=pickle.load(f)
cic_feats=list(d['feature_names']); X=np.asarray(d['X'],float); sc=d.get('scaler',None)
X_orig=X*sc.scale_+sc.mean_ if (sc is not None and hasattr(sc,'scale_')) else X
cic_df=pd.DataFrame(X_orig,columns=cic_feats)
benign=d.get('label_mapping',{}).get('Benign',0); y_cic=(np.asarray(d['y'])!=benign).astype(int)
unsw_tr=pd.read_csv(UNSW_TRAIN); unsw_te=pd.read_csv(UNSW_TEST)
y_utr=unsw_tr['label'].astype(int).values; y_ute=unsw_te['label'].astype(int).values
Xc_all=build_matrix(cic_df,'cic')
Xc_tr_raw,Xc_te_raw,yc_tr,yc_te=train_test_split(Xc_all,y_cic,test_size=0.3,random_state=SEED,stratify=y_cic)
scc=StandardScaler().fit(Xc_tr_raw); Xc_te=scc.transform(Xc_te_raw)
Xu_tr_raw=build_matrix(unsw_tr,'unsw'); Xu_te_raw=build_matrix(unsw_te,'unsw')
scu=StandardScaler().fit(Xu_tr_raw); Xu_te=scu.transform(Xu_te_raw)
print('CIC test:',Xc_te.shape,'| UNSW test:',Xu_te.shape)
print('=== SEL 3 (muat data uji) SELESAI ===')

In [ ]:
def load_models(direction):
    dd=os.path.join(MODELDIR,direction.replace('->','_to_')); models={}
    for v in VARIANTS:
        p=os.path.join(dd,f'{v}.json')
        if os.path.exists(p):
            m=XGBClassifier(); m.load_model(p); models[v]=m
    with open(os.path.join(dd,'scaler.pkl'),'rb') as f: scd=pickle.load(f)
    return models, scd['mean'], scd['scale']

def eval_all(direction, X_src_te,y_src_te, X_tgt_te,y_tgt_te):
    """Untuk tiap varian: clean(source/target) + evasion(unconstrained/functional/adaptive) di TARGET."""
    models, mean, scale = load_models(direction)
    rng=np.random.RandomState(SEED)
    n=min(MAXN,len(X_tgt_te)); idx=rng.choice(len(X_tgt_te),n,replace=False)
    Xt,yt=X_tgt_te[idx],y_tgt_te[idx]
    rows=[]
    for v,m in models.items():
        r={'arah':direction,'model':v}
        r['clean_source']=round(ev(y_src_te,m.predict(X_src_te))['mcc'],4)
        r['clean_target']=round(ev(yt,m.predict(Xt))['mcc'],4)
        # adaptive white-box: saliency dari model m sendiri, functional-preserving
        S=saliency(m,Xt,yt)
        for e in EPS_EVAL:
            Xu=fgsm_unconstrained(Xt,S,e); Xf=fgsm_functional(Xt,S,e,mean,scale)
            r[f'unconstrained_eps{e}']=round(ev(yt,m.predict(Xu))['mcc'],4)
            r[f'adaptive_functional_eps{e}']=round(ev(yt,m.predict(Xf))['mcc'],4)
        rows.append(r)
    return rows
print('=== SEL 4 (fungsi evaluasi) SELESAI ===')

In [ ]:
RESULTS={'features':CANON,'eps_eval':EPS_EVAL,'rows':[]}
RESULTS['rows']+=eval_all('CIC->UNSW',Xc_te,yc_te,Xu_te,y_ute)
RESULTS['rows']+=eval_all('UNSW->CIC',Xu_te,y_ute,Xc_te,yc_te)
dfe=pd.DataFrame(RESULTS['rows'])
import IPython.display as ipd; ipd.display(dfe)
print('=== SEL 5 (jalankan evaluasi 2 arah) SELESAI ===')

In [ ]:
# --- Grouped bar: clean_target vs adaptive_functional_eps0.1 per varian (fokus klaim Paper 2) ---
for direction in ['CIC->UNSW','UNSW->CIC']:
    sub=dfe[dfe['arah']==direction]
    if sub.empty: continue
    labels=sub['model'].tolist(); x=np.arange(len(labels)); w=0.38
    fig,ax=plt.subplots(figsize=(7,3.8))
    ax.bar(x-w/2, sub['clean_target'], w, label='clean (cross-network)', color='#4C72B0')
    ax.bar(x+w/2, sub['adaptive_functional_eps0.1'], w, label='adaptive functional evasion (eps=0.1)', color='#C44E52')
    ax.axhline(0,color='k',lw=0.8); ax.set_xticks(x); ax.set_xticklabels(labels,rotation=15)
    ax.set_ylabel('MCC'); ax.set_title(f'Paper 2 — {direction}: generalization vs evasion robustness')
    ax.legend(fontsize=8); plt.tight_layout()
    p=os.path.join(OUTDIR,f'paper2_{direction.replace("->","_to_")}.png'); plt.savefig(p,bbox_inches='tight'); plt.close()
    print('saved',p)
print('=== SEL 6 (grafik) SELESAI ===')

In [ ]:
dfe.to_csv(os.path.join(OUTDIR,'paper2_eval.csv'),index=False)
json.dump(RESULTS,open(os.path.join(OUTDIR,'paper2_eval_results.json'),'w'),indent=2)
print('tersimpan paper2_eval_results.json + csv')
try:
    import boto3; s3=boto3.client('s3',region_name=REGION); up=0
    for fn in sorted(os.listdir(OUTDIR)):
        if fn.endswith(('.json','.png','.csv')): s3.upload_file(os.path.join(OUTDIR,fn),S3_BUCKET,f'{S3_PREFIX}/paper2_eval/{fn}'); up+=1
    print(f'upload {up} artefak -> s3://{S3_BUCKET}/{S3_PREFIX}/paper2_eval/')
except Exception as e: print('upload gagal:',e)
print('=== SEL 7 (simpan + upload) SELESAI ===')
print('Interpretasi: bandingkan baris fewshot_adv vs fewshot (clean_target harus mirip = generalisasi terjaga)')
print('dan vs adv/baseline (adaptive_functional harus lebih tinggi = lebih tahan evasion).')